# 01. Chạy baseline

Notebook này chạy hai cấu hình nền cho phần deep learning: `frozen` và `full_finetune`. Hai baseline này tạo mốc so sánh để đánh giá LoRA có giữ được hiệu năng tốt trong khi giảm số tham số cần huấn luyện hay không.

Ý nghĩa của hai cấu hình:

- `frozen`: đóng băng encoder, chỉ huấn luyện classifier. Đây là baseline rất nhẹ.
- `full_finetune`: cập nhật toàn bộ model. Đây là baseline mạnh nhưng tốn nhiều tham số trainable nhất.

Sau khi chạy, mỗi thí nghiệm sẽ sinh thư mục `results/runs/<run_name>/` chứa `metrics.json`, `config.resolved.json` và checkpoint tốt nhất.

## Bước 1. Đặt đúng thư mục làm việc

Cell này bảo đảm notebook luôn chạy từ thư mục `deep-learning`, kể cả khi người dùng mở notebook từ thư mục `notebooks`. Nhờ vậy các đường dẫn như `configs/frozen.yaml` và `results/runs` sẽ luôn đúng.

In [ ]:
from pathlib import Path
import os

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
os.chdir(root)
print(f"Thư mục làm việc: {Path.cwd()}")

## Bước 2. Chạy hai baseline

Cell này đọc từng file config, khởi tạo model/dataset theo config rồi gọi `run_training`. Với mỗi cấu hình, notebook in ra:

- `run_name`: tên thí nghiệm.
- `best_acc`: validation accuracy tốt nhất trong quá trình train.
- `trainable_params`: số tham số được cập nhật.

Trong lần chạy đã dùng cho báo cáo, frozen đạt khoảng `0.6433` accuracy với `258` tham số trainable, còn full fine-tune đạt khoảng `0.7661` accuracy với `4,386,178` tham số trainable.

In [ ]:
from src.config import load_config
from src.train import run_training

baseline_configs = [
    "configs/frozen.yaml",
    "configs/full_finetune.yaml",
]

baseline_metrics = []
for config_path in baseline_configs:
    print(f"\nĐang chạy {config_path}")
    metrics = run_training(load_config(config_path))
    baseline_metrics.append(metrics)
    print(
        metrics["run_name"],
        "best_acc=", round(metrics["best_val_accuracy"], 4),
        "trainable_params=", metrics["model"]["trainable_params"],
    )

## Kết quả cần kiểm tra

Sau cell trên, kiểm tra các file sau:

- `results/runs/frozen/metrics.json`
- `results/runs/full_finetune/metrics.json`

Nếu hai file này tồn tại và có `best_val_accuracy`, pipeline baseline đã chạy thành công. Các số này sẽ được dùng làm mốc để so sánh với LoRA trong notebook tiếp theo.